## chain
使用LCEL，可以构造出结构最简单的Chain。
工作
LangChain表达式语言（LCEL，LangChain Expression Language）是一种声明式方法，可以轻松地
将多个组件链接成 AI 工作流。它通过Python原生操作符（如管道符|）将组件连接成可执行流程，显
著简化了AI应用的开发。
LCEL的基本构成：提示（Prompt）+ 模型（Model）+ 输出解析器（OutputParser）


In [53]:
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

llmDS = ChatOpenAI(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

prompt = PromptTemplate.from_template(
    "给我讲一个双关语笑话，语言是中文，主题是关于{topic}的"
)

parser = StrOutputParser()

chain = prompt | llmDS | parser
response = chain.invoke({"topic":"飞机"})

print(response)

当然！这里有一个中文双关语笑话，关于飞机：  

**笑话：**  
有一天，飞机在跑道上准备起飞，突然听到广播说：“各位乘客，抱歉，由于机械故障，我们得‘掉头’检查一下。”  
乘客们很慌：“掉头？飞机怎么掉头？”  
空姐淡定地解释：“别担心，就是‘掉’个头——机长先去理个发。”  

**解析：**  
“掉头”在中文里既可以指“飞机转向”，也可以字面理解为“把头剪掉/换个发型”。这里故意混淆了航空术语和日常理发用语，制造幽默。  

希望这个笑话让你会心一笑！😄


##  LangChain Expression Language顺序链
顺序链允许将多个链顺序连接起来，每个Chain的输出作为下一个Chain的输入，
形成特定场景的流水线（Pipeline）。

In [62]:
praser = StrOutputParser()
promptA = ChatPromptTemplate.from_messages(
    [
        ("system","你是一名资深的中国历史学家,对中华上下五千年的历史都熟悉，并且有自己独特的理解。"),
        ("human","请你介绍一下中国历史上，{dynasty}的主要进程")
    ]
)

chainA = promptA | llmDS | praser
print(chainA.invoke({"dynasty":"明朝"}))

promptB = ChatPromptTemplate.from_messages(
    [
        ("system","你是一名资深的总结专家，擅长对长文本的总结，你的主要工作是将传进来的文本提炼出最简练的部分"),
        ("human","这是针对一个提问的完整的解释说明内容：{description}")
    ]
)

chainB = promptB | llmDS | praser

# 定义 chainA 输出如何映射到 chainB
full_chain = (
    {"dynasty": lambda x: x}  # 提取 input 作为 dynasty
    | promptA
    | llmDS
    | parser
    | {"description": lambda x: x}     # chainA 的输出作为 description
    | promptB
    | llmDS
    | parser
)

response = full_chain.invoke({"dynasty":"明朝"})

print(response)


明朝（1368年－1644年）是中国历史上由汉族建立的大一统王朝，其进程可概括为三个阶段：开国奠基、鼎盛与危机、衰亡与变革。

**开国奠基（洪武至永乐）**  
明太祖朱元璋推翻元朝，定都南京，建立高度中央集权制度，如废除丞相、设立锦衣卫，并推行屯田、抑制豪强，恢复社会经济。明成祖朱棣通过“靖难之役”夺位后迁都北京，派郑和七下西洋，编纂《永乐大典》，并设立内阁、东厂，强化皇权与边疆控制。这一时期，明朝疆域扩大，国力强盛。

**鼎盛与危机（仁宣至嘉靖）**  
“仁宣之治”时期，社会安定，经济繁荣。但中期后，土地兼并、财政危机加剧，土木堡之变（1449年）导致英宗被俘，边防动摇。明武宗、明世宗时期，宦官干政（如刘瑾）、权臣专权（如严嵩）、倭寇侵扰及蒙古威胁交织出现，张居正推行“一条鞭法”改革虽短暂缓解财政困境，但未能根治体制弊端。

**衰亡与变革（万历至崇祯）**  
万历年间，明神宗长期怠政，党争激化（东林党与阉党），辽东后金崛起（萨尔浒之战惨败）。天启时魏忠贤乱政，崇祯帝虽力图挽救，却无力应对大规模农民起义（李自成、张献忠）、财政崩溃及鼠疫灾害。1644年李自成攻破北京，崇祯自缢，明朝灭亡。随后清军入关，南明政权延续至1662年。

明朝的兴衰折射出君主专制强化下的治理困境：边防消耗、经济失衡与制度僵化最终导致其崩溃。这一历程也为后世提供了中央集权与约束权力的重要反思。
明朝（1368-1644）是汉族建立的统一王朝，分创立（洪武永乐）、鼎盛（仁宣至明中期）、中后期危机（嘉靖万历）、灭亡（天启崇祯）四阶段。朱元璋建朝，朱棣迁都北京，郑和下西洋；仁宣之治后宦官专权、土木堡之变；嘉靖万历间倭寇、党争、女真崛起；崇祯时农民起义与清军入侵，1644年李自成破京，明朝亡。其特点是中央集权强、经济文化繁荣，但有宗室与宦官问题。
